# Chapter 9: Offline RL: Learning from static datasets
### **Reinforcement Learning: The Foundational Ideas** by Rahul Shirale

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rshirale/rl-seminal-papers/blob/main/src/part_3_applications/ch09_cql/Chapter9_CQL.ipynb)

This notebook trains Conservative Q-Learning (CQL), from Kumar et al. (2020),
*Conservative Q-Learning for Offline Reinforcement Learning*, on a fixed log of
1,000,000 PointMaze transitions. It never lets the agent act while it learns.
It then trains plain SAC on the same log, which is what happens when an
off-policy algorithm built for online learning is handed logs instead.

**What you will see.** The test is the chapter's *plaza task*: the point mass
starts in the bottom-left cell of a U-shaped maze, the goal is in the top-left
cell, and a single wall sits between them, so the straight line runs into the
wall and the real route goes around the U. In the chapter's runs, after 50,000
steps, SAC reached that goal in none of 10 rollouts and reached only 4 of 20
random goals. CQL reached it in all 10.

The two critics err in opposite directions, and that is the useful part. Every
reward in this task is 0 or 1, so no true Q-value is negative. SAC's average
Q-value on the dataset's own actions climbed to about 4.9 and settled near 3
while its policy failed. CQL's fell below zero, to about −66 at 50,000 steps,
while its policy succeeded. Finally, fitted Q-evaluation estimates each
policy's value from the logs alone, and because PointMaze has a simulator, we
can check that estimate against the return each policy actually earns.


## 1. What runs here, and how long it takes

Most code cells below are files from the repository: chapter 6's `actor.py` and `critic.py`, chapter 9's `cql.py`
(listings 9.1 to 9.4, exactly as printed, one cell per listing), and the two
experiment scripts, `train_cql_vs_sac.py` and `eval_fqe_vs_true.py`. The
cells between them only call those files and plot what they wrote.
`tests/test_ch09_notebook.py` compares each file's cell with the file
character by character, so what you run here and what the chapter prints
cannot drift apart.

The two scripts go through one fixed transformation first, because a notebook
is not a package:

- **Imports from `src.` are dropped**, along with the `L.` and `T.` module
  prefixes they create. Every name they would bind is already defined in this
  notebook by the time the cell runs.
- **`CODE_DIR` is set once, in the setup cell.** The scripts keep their runs
  next to their own file; a notebook has no file, so runs go to `./runs/<name>/`
  in the working directory. Opened from its own folder, the notebook shares
  `runs/` with the scripts, and the repository already ignores that folder.
- **The `if __name__ == "__main__":` lines are dropped.** A notebook kernel runs
  every cell as `__main__`, so with the guard in place the script would start
  parsing the kernel's own command line the moment its cell ran.

**Requirements.** PyTorch, Minari with its download and storage extras, and
Gymnasium Robotics, which provides the PointMaze environment. The dataset is a
one-time download of about 530 MB, cached under `~/.minari/datasets`, and
loading its 1,000,000 transitions takes under a minute.

**Runtime.** A training step takes about 0.04 to 0.05 seconds on a laptop CPU
with 8 threads, so each 50,000-step run takes about 40 minutes, plus its
evaluations. SAC's evaluations take longer than CQL's, because a rollout that
never reaches its goal runs the full 300 steps. Allow 1.5 to 2 hours for
both runs and the FQE check. Finished runs are kept under `runs/<name>/`, and
running the notebook again skips them. For a quick look, set `STEPS = 12_500`
in section 7: in the chapter's test runs, CQL reached the goal in every
evaluation episode by then. Judge a policy only after the full run.

**One rule broken on purpose.** Offline RL forbids running the policy before
it is deployed. PointMaze ships with a simulator, so sections 9 and 10 run the
trained policies in it, the way benchmarks do, to check what the offline
numbers claim. A production system has no such check.


In [ ]:
# Uncomment in Google Colab, which already has torch and matplotlib. Locally,
# run `make install-offline` from the repository root.
# %pip install "minari[hdf5,hf]" gymnasium-robotics

import copy
import csv
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# The repository's test suite sets CH9_FAST=1 to execute this notebook end to
# end in a few minutes. It shortens the training runs and the evaluation; it
# changes nothing about the algorithm. Leave it unset to run the chapter.
FAST = bool(os.environ.get("CH9_FAST"))

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

# The scripts inlined below keep their runs next to their own file. A
# notebook has no file, so its runs go to ./runs/<name>/ instead.
CODE_DIR = os.getcwd()

# The training script defaults to 8 threads. More rarely helps networks
# this small, and fewer cores than threads slows every step.
THREADS = min(8, os.cpu_count() or 1)

print("torch  ", torch.__version__)
print("threads", THREADS)
print("runs   ", os.path.join(CODE_DIR, "runs"))


## 2. Chapter 6's actor and critic, unchanged

Kumar et al. (2020) describe continuous CQL as SAC with the critic's loss
changed, and the code takes that literally: the networks are chapter 6's,
imported rather than rewritten. The actor is the squashed-Gaussian policy. It
returns an action and its log-probability, or, with `deterministic=True`, the
mean action and `None`, which is how every evaluation below runs it. The
critic returns two Q-value estimates, and everything downstream takes their
minimum.

Both are sized for PointMaze: a six-dimensional state (the point's position and
velocity, and the goal's position) and a two-dimensional action, a force in
[−1, 1] along each axis.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

LOG_STD_MIN = -5.0
LOG_STD_MAX = 2.0


class Actor(nn.Module):
    """Stochastic squashed-Gaussian policy, Appendix C of Haarnoja et al. (2018a).

    A shared two-layer backbone feeds separate mean and log-standard-deviation
    heads. The requirement that the actor be *stochastic* is mathematical, not
    a convenience: a deterministic policy is a Dirac delta, its entropy is
    negative infinity, and the entropy term in the maximum entropy objective is
    then undefined. SAC cannot use a deterministic actor.

    Note that ``log_std`` is a head, so it depends on the state. Chapter 5's
    PPO actor carried a state-independent learned scalar instead. A
    state-dependent spread lets SAC be exploratory in unfamiliar regions of the
    state space and precise in well-understood ones.

    Sampling uses the reparameterization trick -- ``rsample`` draws
    ``u = mu + sigma * eps`` with ``eps ~ N(0, I)`` treated as a constant, so
    the path from the action back to the weights stays differentiable -- and
    ``tanh`` bounds the result without the hard clipping that would kill the
    gradient at the boundary.

    The returned log-probability carries the tanh change-of-variables
    correction. Omitting it is the most common SAC implementation bug: the
    entropy signal that drives both the actor loss and the temperature update
    is then biased, and the policy may fail to explore or diverge outright.
    """

    def __init__(self, state_dim: int, action_dim: int, hidden: int = 256,
                 max_action: float = 1.0):
        super().__init__()
        self.max_action = max_action
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.mean_head = nn.Linear(hidden, action_dim)
        self.log_std_head = nn.Linear(hidden, action_dim)

    def forward(self, state: torch.Tensor, deterministic: bool = False):
        """Returns ``(action, log_prob)``.

        With ``deterministic=True`` the mean action ``tanh(mu) * max_action``
        is returned and ``log_prob`` is ``None``. That is the evaluation path:
        the policy is trained against the entropy-augmented objective but
        scored against the plain reward, so the mean action typically returns
        slightly more than a sample does (Haarnoja et al., 2018a).
        """
        x = self.net(state)
        mean = self.mean_head(x)
        # Clamped for numerical stability: an unclamped head can drive sigma to
        # zero (log-probabilities to infinity) or explode it.
        log_std = self.log_std_head(x).clamp(LOG_STD_MIN, LOG_STD_MAX)
        std = log_std.exp()

        if deterministic:
            return torch.tanh(mean) * self.max_action, None

        dist = Normal(mean, std)
        u = dist.rsample()                 # reparameterized: u = mu + sigma*eps
        action = torch.tanh(u)             # squashed to (-1, 1)

        log_prob = dist.log_prob(u).sum(-1, keepdim=True)
        # log pi(a|s) = log p(u|s) - sum_i log(1 - tanh^2(u_i)).
        #
        # The second term is the log absolute Jacobian determinant of tanh.
        # `2 * (log 2 - u - softplus(-2u))` is an algebraically identical but
        # numerically stable form of `log(1 - tanh^2(u))`, which underflows to
        # log(0) once |u| is large; it is the form the authors' own
        # implementation uses. `device=u.device` keeps the constant on the same
        # hardware as the network, so this runs unchanged on CPU or GPU.
        correction = (
            2 * (torch.log(torch.tensor(2.0, device=u.device))
                 - u - F.softplus(-2 * u))
        ).sum(-1, keepdim=True)
        log_prob = log_prob - correction

        return action * self.max_action, log_prob


In [ ]:
import torch
import torch.nn as nn


class Critic(nn.Module):
    """
    Twin soft Q-networks from Haarnoja et al. (2018), "Soft Actor-Critic."

    Two independently initialized Q-networks Q1 and Q2 map (state, action)
    pairs to scalar soft Q-values. SAC uses the minimum of the two as the
    Bellman target — the clipped double-Q trick of Fujimoto et al. (2018) —
    to counteract the Q-value overestimation that would otherwise let the
    actor exploit inflated estimates.

    forward() returns both Q-values so the caller can apply min(Q1, Q2)
    without crossing the class boundary twice.
    """

    def __init__(self, state_dim: int, action_dim: int, hidden: int = 256):
        super().__init__()

        def mlp():
            return nn.Sequential(
                nn.Linear(state_dim + action_dim, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, 1),
            )

        self.q1 = mlp()
        self.q2 = mlp()

    def forward(self, state: torch.Tensor, action: torch.Tensor):
        sa = torch.cat([state, action], dim=-1)
        return self.q1(sa), self.q2(sa)


## 3. Listing 9.1: the conservative critic loss

The loss has three parts, and the code mirrors them line for line:

1. **The TD loss** (callout 1): the standard Bellman error on the dataset's own
   transitions, for both critics.
2. **Push down** (callout 2): at every dataset state, a soft maximum of the
   Q-values over all actions. A soft maximum is dominated by its largest terms,
   so this lands hardest on whichever actions the critic currently overvalues.
3. **Push up** (callout 3): the Q-values of the actions actually in the data.

With continuous actions, the sum over actions inside the log-sum-exp is an
integral, so it is estimated by importance sampling: 10 uniform actions and 10
policy actions per state, each Q-value corrected by subtracting the
log-density of the distribution it came from. The uniform density is 0.5 per
dimension on [−1, 1], so `log_unif` is `2 * log(0.5)` here.

Three details are easy to miss:

- The target `y` has **no entropy term**. Chapter 6's SAC subtracted it; the
  CQL objective uses the plain Bellman backup and applies entropy only in the
  actor's update (listing 9.3), as the authors' released code does by default.
- The TD loss leaves out the ½ in front of the squared error, again as the
  authors' code does, so `cql_alpha=5.0` here is the same setting as the
  paper's α = 5.0.
- Chapter 6 used α for the entropy temperature and the CQL paper uses α for the
  conservative weight, so the code says `cql_alpha` for one and `ent_alpha` for
  the other.


In [ ]:
"""Listings 9.1-9.4 of chapter 9: the continuous CQL critic loss, the offline
buffer and training loop, and fitted Q-evaluation, exactly as printed."""
# --- Listing 9.1 ---------------------------------------------------------
import math
import torch
import torch.nn.functional as F

def soft_lse(q_rand, q_pi, log_pi, log_unif):
    cat = torch.cat([q_rand - log_unif, q_pi - log_pi], 1)
    return torch.logsumexp(cat, dim=1).mean()

def cql_critic_loss(critic, critic_target, actor, batch,
                    gamma=0.99, cql_alpha=5.0, n=10):
    s, a, r, s2, d = batch
    b, act_dim = a.shape

    with torch.no_grad():
        a2, _ = actor(s2)
        q1_t, q2_t = critic_target(s2, a2)
        y = r + gamma * (1 - d) * torch.min(q1_t, q2_t)

    q1, q2 = critic(s, a)
    td_loss = F.mse_loss(q1, y) + F.mse_loss(q2, y)  # <1>

    s_rep = s.repeat_interleave(n, dim=0)
    with torch.no_grad():
        a_rand = torch.rand(b * n, act_dim,
                            device=s.device) * 2 - 1
        a_pi, logp_pi = actor(s_rep)
    log_unif = act_dim * math.log(0.5)
    logp_pi = logp_pi.view(b, n)

    q1_r, q2_r = critic(s_rep, a_rand)
    q1_p, q2_p = critic(s_rep, a_pi)
    push_down = (                                    # <2>
        soft_lse(q1_r.view(b, n), q1_p.view(b, n),
                 logp_pi, log_unif)
        + soft_lse(q2_r.view(b, n), q2_p.view(b, n),
                   logp_pi, log_unif))
    push_up = q1.mean() + q2.mean()                  # <3>

    return td_loss + cql_alpha * (push_down - push_up)  # <4>


## 4. Listing 9.2: a fixed dataset

`D4RL/pointmaze/umaze-v2` holds 1,000,000 transitions across 13,210 episodes.
`minari show D4RL/pointmaze/umaze-v2` says how they were collected: a PD
controller follows waypoints from a planner (QIteration), with random noise
added to its actions. The planner routes around the walls, so the logged
actions mostly follow the corridors, and pushing straight at a wall toward a
goal on the other side is exactly what the data rarely contains.

The buffer is a set of preallocated tensors with random index sampling rather
than chapter 6's `ReplayBuffer`, which is built for transitions streaming in.
An offline dataset arrives all at once and never changes. The buffer also
keeps every episode's first observation in `buffer.starts`; training ignores
them, and FQE in listing 9.4 averages over them.


In [ ]:
# --- Listing 9.2 ---------------------------------------------------------
import minari
import numpy as np

class OfflineBuffer:
    def __init__(self, *columns):
        self.cols = [torch.as_tensor(np.concatenate(c),
                                     dtype=torch.float32)
                     for c in columns]

    def sample(self, batch_size):
        idx = torch.randint(len(self.cols[0]), (batch_size,))
        return [c[idx] for c in self.cols]

def load_minari(dataset_id="D4RL/pointmaze/umaze-v2"):
    dataset = minari.load_dataset(dataset_id,
                                  download=True)     # <1>
    s, a, r, s2, d, starts = [], [], [], [], [], []
    for ep in dataset.iterate_episodes():
        obs = np.concatenate(                        # <2>
            [ep.observations["observation"],
             ep.observations["desired_goal"]], axis=1)
        starts.append(obs[:1])
        s.append(obs[:-1])
        s2.append(obs[1:])
        a.append(ep.actions)
        r.append(ep.rewards[:, None])
        d.append(ep.terminations[:, None])
    buffer = OfflineBuffer(s, a, r, s2, d)
    buffer.starts = OfflineBuffer(starts).cols[0]
    return buffer


In [ ]:
buffer = load_minari()
s, a, r, s2, d = buffer.cols

print(f"transitions        {len(s):>9,}")
print(f"episodes (starts)  {len(buffer.starts):>9,}")
print(f"state, action dims {s.shape[1]:>6}, {a.shape[1]}")
print(f"action range       [{a.min():.2f}, {a.max():.2f}]")
print(f"mean reward        {r.mean():9.4f}")
print(f"terminations       {int(d.sum()):>9,}")


Two numbers above shape everything that follows. The **mean reward** is about
0.013 per step: the reward is 1 at the goal and 0 everywhere else, so the
signal is sparse. And the **termination count** is zero. The data was collected
as a continuing task, where reaching the goal draws a new goal rather than
ending the episode, so every Bellman target bootstraps from the next state.

Listing 9.1 can run on one real batch already, with untrained networks. At
`cql_alpha=0` it is the plain TD loss, and the loss is linear in `cql_alpha`.
That linearity is what lets the training script in section 7 read the
penalty off a fixed probe batch as the difference of two calls.


In [ ]:
critic, actor = Critic(6, 2), Actor(6, 2)
critic_target = copy.deepcopy(critic)
batch = buffer.sample(256)

for cql_alpha in (0.0, 1.0, 5.0):
    torch.manual_seed(1)  # the same random and policy actions every time
    loss = cql_critic_loss(critic, critic_target, actor, batch,
                           cql_alpha=cql_alpha)
    print(f"cql_alpha {cql_alpha:3.1f}   loss {loss.item():8.4f}")


## 5. Listing 9.3: training without the environment

Apart from the missing environment, this is chapter 6's SAC update with the
critic loss swapped for CQL's, which is the paper's own description of the
method. The one other change Kumar et al. (2020) make to SAC is the policy
learning rate: 3e-5 rather than 3e-4, so the actor moves slowly relative to the
critic it optimizes against. The SAC temperature is held at 0.2 and acts only
on the actor, through `ent_alpha * log_pi` at callout 2.

**Why the critic's target leaves out entropy.** Keeping chapter 6's target
fails on PointMaze. As CQL draws the actor toward the dataset's actions, the
policy turns nearly deterministic, its log-probabilities climb to about 8, and
every backup subtracts about 0.2 × 8 = 1.6, over a hundred times the average
reward. In the chapter's test run at a conservative weight of 5, the average
Q-value on dataset actions sank to −199 after 50,000 steps and was still
falling. With the entropy term removed it ends at −66, a third of the drop.


In [ ]:
# --- Listing 9.3 ---------------------------------------------------------
import copy

def soft_update(target, source, tau=0.005):
    for tp, sp in zip(target.parameters(), source.parameters()):
        tp.data.lerp_(sp.data, tau)

def train_offline_cql(actor, critic, buffer, steps=50_000,
                      batch_size=256, cql_alpha=5.0,
                      ent_alpha=0.2):
    critic_target = copy.deepcopy(critic)
    critic_opt = torch.optim.Adam(critic.parameters(), lr=3e-4)
    actor_opt = torch.optim.Adam(actor.parameters(),
                                 lr=3e-5)            # <1>

    for step in range(steps):
        batch = buffer.sample(batch_size)
        q_loss = cql_critic_loss(critic, critic_target, actor,
                                 batch, cql_alpha=cql_alpha)
        critic_opt.zero_grad()
        q_loss.backward()
        critic_opt.step()

        s = batch[0]
        a_new, log_pi = actor(s)
        q1, q2 = critic(s, a_new)
        pi_loss = (ent_alpha * log_pi
                   - torch.min(q1, q2)).mean()      # <2>
        actor_opt.zero_grad()
        pi_loss.backward()
        actor_opt.step()

        soft_update(critic_target, critic)
        if step % 10_000 == 0:
            print(f"step {step:>7} | Q loss {q_loss.item():8.2f}"
                  f" | pi loss {pi_loss.item():8.2f}")


The chapter puts the pieces together in three lines. They are not run here,
because section 7's script runs the same update inline, which lets it
evaluate and checkpoint between steps:

```python
buffer = load_minari()
actor, critic = Actor(6, 2), Critic(6, 2)
train_offline_cql(actor, critic, buffer)
```


## 6. Listing 9.4: fitted Q-evaluation

An offline agent cannot be tested by running it. Fitted Q-evaluation (Le et
al., 2019) estimates its value from the logs instead: freeze the trained
policy, train a fresh Q-network on the dataset's transitions with a target that
asks the *policy*, not the dataset, for the next action (callout 1), then
average the learned values over the episode start states that listing 9.2
saved (callout 3).

The listing keeps chapter 6's minimum over two critics. That makes the
estimate biased low but steadier. In the chapter's run, a CQL policy with a
true discounted return of 1.27 got an FQE estimate of 0.32 after 100,000
steps, yet the estimates for its 50 start states tracked their true returns
with a correlation of 0.76.

FQE bootstraps through the new policy's actions, so wherever that policy
departs from the logged behavior, the evaluator extrapolates just as a naive
critic does. Treat a high estimate for a policy that rarely agrees with the
data as a reason to look closer.


In [ ]:
# --- Listing 9.4 ---------------------------------------------------------
def fitted_q_evaluation(policy, buffer, fqe_critic,
                        steps=100_000, gamma=0.99):
    target = copy.deepcopy(fqe_critic)
    opt = torch.optim.Adam(fqe_critic.parameters(), lr=3e-4)
    policy.eval()

    for _ in range(steps):
        s, a, r, s2, d = buffer.sample(256)
        with torch.no_grad():
            a2, _ = policy(s2, deterministic=True)   # <1>
            q1_t, q2_t = target(s2, a2)
            y = r + gamma * (1 - d) * torch.min(q1_t, q2_t)
        q1, q2 = fqe_critic(s, a)                    # <2>
        loss = F.mse_loss(q1, y) + F.mse_loss(q2, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        soft_update(target, fqe_critic)

    with torch.no_grad():                            # <3>
        s0 = buffer.starts
        a0, _ = policy(s0, deterministic=True)
        return torch.min(*fqe_critic(s0, a0)).mean()


## 7. The experiment: CQL against SAC on the same logs

`train_cql_vs_sac.py` runs listing 9.3's update inline and adds the
measurements the chapter's figures come from:

- every `--log-every` steps (5,000), on a fixed probe of 4,096 dataset
  transitions: the average dataset Q-value, and the critic loss split into its
  TD part and its CQL penalty (`push_down - push_up`, summed over both
  critics, before `cql_alpha`), to `progress.csv`;
- every `--eval-every` steps (12,500): 10 rollouts of the plaza task and 20 to
  random goals, 300 steps each, to `log.csv`, with an actor-and-critic
  checkpoint and the plaza paths.

SAC here is CQL with `cql_alpha=0`, which is exercise 9.1. Like listing 9.1,
that baseline keeps the entropy term out of the critic target, so it is SAC
without the entropy backup.


In [ ]:
"""Trains CQL and plain SAC offline on PointMaze UMaze for chapter 9's figure.

Both runs use listings 9.1 and 9.2 unchanged (cql.py) and run listing 9.3's
update inline, so the script can evaluate and checkpoint between steps; the
SAC run is CQL with cql_alpha=0, which is exercise 9.1. Like listing 9.1,
that baseline keeps the entropy term out of the critic target, so it is SAC
without the entropy backup. Every --eval-every steps the script logs the
mean Q-value the critic assigns to dataset actions, evaluates the policy in
the recovered environment, and saves a checkpoint.

The "plaza" evaluation starts the point in the bottom-left cell and puts the
goal in the top-left cell. A single wall separates them, so the straight-line
route goes through the wall and the real route goes around the U.

Every --log-every steps it also appends to progress.csv the losses, the
measured seconds per training step (evaluation excluded), and, on a fixed
probe of 4,096 dataset transitions, the average dataset Q-value with the
critic loss split into its TD part and its CQL penalty (push_down - push_up,
summed over both critics, before cql_alpha). Those are the curves exercise
9.6 asks for. Every evaluation also keeps a checkpoint_<step>.pt with the
actor and critic, so the checkpoint at the dataset Q-value's peak can be
evaluated later; checkpoint.pt is always the latest full checkpoint, for
--resume.

Usage, from the repository root:
    python -m src.part_3_applications.ch09_cql.train_cql_vs_sac \
        --name cql --cql-alpha 5.0
    python -m src.part_3_applications.ch09_cql.train_cql_vs_sac \
        --name sac --cql-alpha 0.0

Exercise 9.6: train at alpha 1 for 50,000 steps, then continue the same run
to 200,000. --resume reads runs/<name>/checkpoint.pt and stops if
--cql-alpha differs from the run's, so repeat the same --name and alpha.
    python -m src.part_3_applications.ch09_cql.train_cql_vs_sac \
        --name cql_a1 --cql-alpha 1.0
    python -m src.part_3_applications.ch09_cql.train_cql_vs_sac \
        --name cql_a1 --cql-alpha 1.0 --steps 200000 --resume
"""
import argparse
import copy
import csv
import os
import time

import numpy as np
import torch


DATASET_ID = "D4RL/pointmaze/umaze-v2"
PLAZA_START, PLAZA_GOAL = [3, 1], [1, 1]


def make_eval_env():
    dataset = minari.load_dataset(DATASET_ID)
    return dataset.recover_environment(
        continuing_task=False, reset_target=False, max_episode_steps=300)


def obs_vec(obs):
    return np.concatenate([obs["observation"], obs["desired_goal"]])


def rollout(env, actor, seed, options=None):
    obs, _ = env.reset(seed=seed, options=options)
    path, success = [obs["observation"][:2].copy()], False
    goal = obs["desired_goal"].copy()
    for _ in range(300):
        s = torch.as_tensor(obs_vec(obs), dtype=torch.float32)[None]
        with torch.no_grad():
            a, _ = actor(s, deterministic=True)
        obs, _, term, trunc, info = env.step(a[0].numpy())
        path.append(obs["observation"][:2].copy())
        success = success or bool(info.get("success"))
        if term or trunc:
            break
    return np.array(path), goal, success


def evaluate(env, actor, n_random=20):
    plaza = [rollout(env, actor, seed=i,
                     options={"reset_cell": PLAZA_START,
                              "goal_cell": PLAZA_GOAL})
             for i in range(10)]
    rand = [rollout(env, actor, seed=1000 + i) for i in range(n_random)]
    return plaza, rand


def probe_metrics(critic, critic_target, actor, probe):
    """Dataset Q, TD loss, and CQL penalty on the fixed probe batch.

    Calls listing 9.1 twice with identical random draws: cql_alpha=0 gives
    the TD loss alone, cql_alpha=1 adds the penalty once. The forked RNG
    leaves training's random stream, and so --resume, untouched.
    """
    with torch.no_grad(), torch.random.fork_rng():
        dq = torch.min(*critic(probe[0], probe[1])).mean().item()
        torch.manual_seed(0)
        td = cql_critic_loss(critic, critic_target, actor, probe,
                               cql_alpha=0.0).item()
        torch.manual_seed(0)
        full = cql_critic_loss(critic, critic_target, actor, probe,
                                 cql_alpha=1.0).item()
    return dq, td, full - td


def truncate_log(path, last_step):
    """Drops rows logged after last_step, which a resumed run logs again."""
    with open(path, newline="") as f:
        rows = list(csv.reader(f))
    with open(path, "w", newline="") as f:
        csv.writer(f).writerows(
            rows[:1] + [r for r in rows[1:] if int(r[0]) <= last_step])


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--name", required=True)
    p.add_argument("--cql-alpha", type=float, required=True)
    p.add_argument("--steps", type=int, default=50_000,
                   help="total steps to reach (counting resumed ones)")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--threads", type=int, default=8)
    p.add_argument("--log-every", type=int, default=5_000)
    p.add_argument("--eval-every", type=int, default=12_500)
    p.add_argument("--resume", action="store_true",
                   help="continue from runs/<name>/checkpoint.pt")
    args = p.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    torch.set_num_threads(args.threads)
    out = os.path.join(CODE_DIR, "runs", args.name)
    os.makedirs(out, exist_ok=True)

    buffer = load_minari(DATASET_ID)
    env = make_eval_env()
    actor, critic = Actor(6, 2), Critic(6, 2)
    critic_target = copy.deepcopy(critic)
    critic_opt = torch.optim.Adam(critic.parameters(), lr=3e-4)
    actor_opt = torch.optim.Adam(actor.parameters(), lr=3e-5)
    probe = buffer.sample(4096)

    ckpt_path = os.path.join(out, "checkpoint.pt")
    log_path = os.path.join(out, "log.csv")
    progress_path = os.path.join(out, "progress.csv")
    start = 0
    if args.resume:
        ck = torch.load(ckpt_path)
        saved = ck.get("cql_alpha")  # absent in older checkpoints
        if saved is not None and saved != args.cql_alpha:
            raise SystemExit(f"{ckpt_path} was trained with --cql-alpha "
                             f"{saved}, not {args.cql_alpha}")
        actor.load_state_dict(ck["actor"])
        critic.load_state_dict(ck["critic"])
        critic_target.load_state_dict(ck["critic_target"])
        critic_opt.load_state_dict(ck["critic_opt"])
        actor_opt.load_state_dict(ck["actor_opt"])
        torch.set_rng_state(ck["torch_rng"])
        start = ck["step"] + 1
        truncate_log(log_path, ck["step"])
        truncate_log(progress_path, ck["step"])
    else:
        with open(log_path, "w", newline="") as f:
            csv.writer(f).writerow(
                ["step", "q_loss", "pi_loss", "dataset_q", "plaza_success",
                 "random_success", "elapsed_s"])
        with open(progress_path, "w", newline="") as f:
            csv.writer(f).writerow(["step", "q_loss", "pi_loss",
                                    "sec_per_step", "elapsed_s",
                                    "dataset_q", "td_loss",
                                    "cql_penalty"])

    t0 = time.time()
    train_time, timed_steps = 0.0, 0
    for step in range(start, args.steps + 1):
        t_step = time.time()
        batch = buffer.sample(256)
        q_loss = cql_critic_loss(critic, critic_target, actor, batch,
                                   cql_alpha=args.cql_alpha)
        critic_opt.zero_grad()
        q_loss.backward()
        critic_opt.step()

        a_new, log_pi = actor(batch[0])
        q1, q2 = critic(batch[0], a_new)
        pi_loss = (0.2 * log_pi - torch.min(q1, q2)).mean()
        actor_opt.zero_grad()
        pi_loss.backward()
        actor_opt.step()
        soft_update(critic_target, critic)
        train_time += time.time() - t_step
        timed_steps += 1

        if step > start and step % args.log_every == 0:
            sps = train_time / timed_steps
            row = [step, round(q_loss.item(), 4), round(pi_loss.item(), 4),
                   round(sps, 4), round(time.time() - t0)]
            row += [round(x, 4) for x in
                    probe_metrics(critic, critic_target, actor, probe)]
            with open(progress_path, "a", newline="") as f:
                csv.writer(f).writerow(row)
            print("progress", " | ".join(str(x) for x in row), flush=True)
            train_time, timed_steps = 0.0, 0

        if step % args.eval_every == 0 or step == args.steps:
            with torch.no_grad():
                dq = torch.min(*critic(probe[0], probe[1])).mean().item()
            plaza, rand = evaluate(env, actor)
            ps = np.mean([r[2] for r in plaza])
            rs = np.mean([r[2] for r in rand])
            row = [step, q_loss.item(), pi_loss.item(), dq, ps, rs,
                   round(time.time() - t0)]
            with open(log_path, "a", newline="") as f:
                csv.writer(f).writerow(row)
            print(" | ".join(str(x) for x in row), flush=True)
            torch.save({"actor": actor.state_dict(),
                        "critic": critic.state_dict(),
                        "critic_target": critic_target.state_dict(),
                        "critic_opt": critic_opt.state_dict(),
                        "actor_opt": actor_opt.state_dict(),
                        "torch_rng": torch.get_rng_state(),
                        "cql_alpha": args.cql_alpha,
                        "step": step}, ckpt_path)
            torch.save({"actor": actor.state_dict(),
                        "critic": critic.state_dict(),
                        "step": step},
                       os.path.join(out, f"checkpoint_{step:07d}.pt"))
            np.savez(os.path.join(out, f"paths_{step:07d}.npz"),
                     plaza=np.array([r[0] for r in plaza], dtype=object),
                     plaza_goal=plaza[0][1])


The cell below trains both runs: SAC, and CQL at the listings' default
`cql_alpha` of 5.0. Each takes about 40 minutes at 50,000 steps (section 1 has
the details). A run already trained to `STEPS` is skipped, and a shorter one
is resumed with the script's own `--resume`, which stops if `cql_alpha` differs
from the run's. Raising `STEPS` and running the cell again continues both runs.


In [ ]:
# eval_fqe_vs_true.py, inlined in section 10, defines a main of its own.
# Hold on to this one before that cell rebinds the name.
train = main

STEPS, EVAL_EVERY, LOG_EVERY = 50_000, 12_500, 5_000
if FAST:
    STEPS, EVAL_EVERY, LOG_EVERY = 200, 100, 100

RUNS = {"sac": 0.0, "cql": 5.0}  # run name -> cql_alpha


def run(name, cql_alpha, steps=STEPS):
    """Trains runs/<name>/ to `steps`: fresh, resumed, or skipped if done."""
    argv = ["--name", name, "--cql-alpha", str(cql_alpha),
            "--steps", str(steps), "--eval-every", str(EVAL_EVERY),
            "--log-every", str(LOG_EVERY), "--seed", str(SEED),
            "--threads", str(THREADS)]
    ckpt = os.path.join(CODE_DIR, "runs", name, "checkpoint.pt")
    if os.path.exists(ckpt):
        done = torch.load(ckpt)["step"]
        if done >= steps:
            print(f"{name}: already trained to step {done:,}; skipping")
            return
        argv.append("--resume")
    # The script reads its options from the command line.
    saved, sys.argv = sys.argv, ["train_cql_vs_sac.py", *argv]
    try:
        train()
    finally:
        sys.argv = saved


for name, cql_alpha in RUNS.items():
    run(name, cql_alpha)


## 8. What each critic believed

Figure 9.7 in the chapter plots the first panel below for the same two runs.
With no environment to test against, these curves are the closest thing
offline RL has to a validation loss (Kumar et al., 2021).

- **Dataset Q-value.** No true Q-value in this task is below 0. SAC's estimate
  climbs while its policy fails: its critic bootstraps from the actions its
  actor picks at the next state, which the logs rarely contain, so this is
  extrapolation error on our own task. CQL's errs the other way: it falls below
  zero and is still falling at 50,000 steps. A conservative estimate may sit
  below zero. The chapter's 200,000-step run at `cql_alpha=1` levels off near
  −42 from about 130,000 steps on.
- **CQL penalty.** The soft maximum over all actions minus the dataset
  actions' Q-values. SAC never trains on it, so it shows how far above the
  data its critic rates actions the data does not contain: about 8.7 to 8.8
  throughout the chapter's run. CQL drives it down, and at α = 5 the chapter's
  run ends with it below zero. The workflow in Kumar et al. (2021) reads a
  penalty that a larger model cannot bring down as a sign to raise α until the
  penalty ends below zero.
- **TD loss.** The Bellman error alone. It does not say which critic is right.


In [ ]:
SAC_C, CQL_C = "#e8590c", "#7048e8"  # the chapter's orange and purple
STYLE = {"sac": ("SAC", SAC_C, (0, (4, 2.5))),
         "cql": (f"CQL (α = {RUNS['cql']:g})", CQL_C, "solid")}


def read_csv(name, file):
    path = os.path.join(CODE_DIR, "runs", name, file)
    with open(path, newline="") as f:
        return [{k: float(v) for k, v in row.items()}
                for row in csv.DictReader(f)]


fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for name in RUNS:
    label, color, style = STYLE[name]
    first = read_csv(name, "log.csv")[0]
    rows = read_csv(name, "progress.csv")
    steps = [first["step"]] + [r["step"] for r in rows]
    panels = [[first["dataset_q"]] + [r["dataset_q"] for r in rows],
              [r["td_loss"] for r in rows],
              [r["cql_penalty"] for r in rows]]
    for ax, ys in zip(axes, panels):
        xs = steps if len(ys) == len(steps) else steps[1:]
        ax.plot(xs, ys, color=color, linestyle=style, linewidth=2,
                marker="o", markersize=4, label=label)

titles = ["Average Q-value on dataset actions", "TD loss (probe batch)",
          "CQL penalty, before cql_alpha"]
for ax, title in zip(axes, titles):
    ax.set_title(title, fontsize=10, loc="left")
    ax.set_xlabel("training steps", fontsize=9)
    ax.grid(color="#e9ecef", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
for ax in (axes[0], axes[2]):
    ax.axhline(0, color="#1e1e1e", linewidth=1)
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()


## 9. The plaza task

Every evaluation ran the deterministic policy in the simulator that
`dataset.recover_environment()` returns, which offline training never touched.
The table lists each evaluation, and the figure draws the 10 plaza rollouts
from the last one, as figure 9.6 in the chapter does. The star marks the goal
cell; each rollout's goal sits at a slightly different spot within it.

In the chapter's runs, SAC slides along the underside of the wall, follows the
corridor up the right side of the U, and ends every rollout pressed into the
far corner, short of the goal. CQL follows the same corridor but turns along
the top row and reaches the goal. Your runs will differ in detail, because
results vary across machines and PyTorch builds even with the same seed.


In [ ]:
print(f"{'run':<5} {'step':>7} {'dataset Q':>10} {'plaza':>7} {'random':>7}")
for name in RUNS:
    for row in read_csv(name, "log.csv"):
        print(f"{name:<5} {int(row['step']):>7,} {row['dataset_q']:>10.2f} "
              f"{row['plaza_success'] * 10:>4.0f}/10 "
              f"{row['random_success'] * 20:>4.0f}/20")


In [ ]:
from matplotlib.patches import Rectangle

# The U-maze interior: three by three cells of side 1, centred on the origin,
# with the left two cells of the middle row walled off.
HALF, INNER_WALL = 1.5, (-1.5, -0.5, 2.0, 1.0)  # wall: x, y, width, height
START, GOAL = (-1.0, -1.0), (-1.0, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
for ax, name in zip(axes, RUNS):
    label, color, style = STYLE[name]
    last = read_csv(name, "log.csv")[-1]
    step = int(last["step"])
    z = np.load(os.path.join(CODE_DIR, "runs", name, f"paths_{step:07d}.npz"),
                allow_pickle=True)
    ax.add_patch(Rectangle((-HALF, -HALF), 2 * HALF, 2 * HALF, fill=False,
                           edgecolor="#adb5bd", linewidth=6))
    ax.add_patch(Rectangle(INNER_WALL[:2], *INNER_WALL[2:],
                           facecolor="#dee2e6", edgecolor="#adb5bd"))
    for path in z["plaza"]:
        path = np.asarray(path, dtype=float)
        ax.plot(path[:, 0], path[:, 1], color=color, linestyle=style,
                linewidth=1.3, alpha=0.85)
        ax.plot(*path[-1], marker="o", markersize=4, color=color)
    ax.plot(*START, marker="o", markersize=9, markerfacecolor="white",
            markeredgecolor="#1e1e1e")
    ax.plot(*GOAL, marker="*", markersize=15, color="#1e1e1e")
    ax.set_title(f"{label} at step {step:,}: "
                 f"{last['plaza_success'] * 10:.0f} of 10 reach the goal",
                 fontsize=10)
    ax.set_xlim(-HALF - 0.08, HALF + 0.08)
    ax.set_ylim(-HALF - 0.08, HALF + 0.08)
    ax.set_aspect("equal")
    ax.axis("off")
fig.tight_layout()
plt.show()


## 10. Checking FQE against the truth

`eval_fqe_vs_true.py` is exercise 9.2's check. For each trained policy it fits
FQE with listing 9.4, then rolls the policy out in the simulator from 50 start
states and compares, state by state, FQE's predicted discounted return with
the discounted return the policy actually earns. The environment uses the
dataset's own settings (a continuing task with a new goal each time the goal
is reached), so both numbers describe the same problem, and rollouts run 800
steps, long enough for γ<sup>t</sup> to be negligible at the end.
`fqe_single_critic` is the same fit with `q1` alone in place of the twin
minimum, the comparison the exercise suggests.


In [ ]:
"""Checks fitted Q-evaluation (listing 9.4) against measured returns.

For each trained agent (runs/<name>/checkpoint.pt) this script fits FQE on
the PointMaze dataset with the chapter's own listing, then compares, start
state by start state, FQE's predicted discounted return with the discounted
return the policy actually earns in the environment from that same state.

The environment uses the dataset's own settings (continuing task, a new
random goal each time the goal is reached), so both numbers describe the
same MDP. Rollouts run long enough that gamma**t is negligible at the end.

--single-critic fits and reads q1 alone instead of listing 9.4's minimum
over twin critics, the comparison exercise 9.2 suggests. Its results go to
fqe_vs_true_single.json unless --out says otherwise.

Usage, from the repository root:
    python -m src.part_3_applications.ch09_cql.eval_fqe_vs_true \
        cql sac --fqe-steps 100000 --starts 50
    python -m src.part_3_applications.ch09_cql.eval_fqe_vs_true \
        cql sac --fqe-steps 100000 --starts 50 --single-critic
"""
import argparse
import copy
import json
import os

import numpy as np
import torch
import torch.nn.functional as F


GAMMA = 0.99


def true_returns(env, actor, seeds, horizon):
    starts, returns = [], []
    for seed in seeds:
        obs, _ = env.reset(seed=int(seed))
        starts.append(obs_vec(obs))
        g, disc = 0.0, 1.0
        for _ in range(horizon):
            s = torch.as_tensor(obs_vec(obs), dtype=torch.float32)[None]
            with torch.no_grad():
                a, _ = actor(s, deterministic=True)
            obs, r, term, trunc, _ = env.step(a[0].numpy())
            g += disc * float(r)
            disc *= GAMMA
            if term or trunc:
                break
        returns.append(g)
    return np.array(starts, dtype=np.float32), np.array(returns)


def fqe_single_critic(policy, buffer, fqe_critic, steps, gamma=GAMMA):
    """Listing 9.4 with q1 alone in the target, the loss, and the estimate.

    fqe_critic is still chapter 6's twin Critic, so q1 starts from the same
    weights as in the twin-minimum fit; q2 gets no gradient and never moves.
    """
    target = copy.deepcopy(fqe_critic)
    opt = torch.optim.Adam(fqe_critic.parameters(), lr=3e-4)
    policy.eval()

    for _ in range(steps):
        s, a, r, s2, d = buffer.sample(256)
        with torch.no_grad():
            a2, _ = policy(s2, deterministic=True)
            q1_t, _ = target(s2, a2)
            y = r + gamma * (1 - d) * q1_t
        q1, _ = fqe_critic(s, a)
        loss = F.mse_loss(q1, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        soft_update(target, fqe_critic)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("runs", nargs="+")
    p.add_argument("--fqe-steps", type=int, default=100_000)
    p.add_argument("--starts", type=int, default=50)
    p.add_argument("--horizon", type=int, default=800)
    p.add_argument("--threads", type=int, default=8)
    p.add_argument("--single-critic", action="store_true",
                   help="use q1 alone instead of the twin minimum")
    p.add_argument("--out", default=None,
                   help="results file name under runs/ (default "
                        "fqe_vs_true.json, or fqe_vs_true_single.json "
                        "with --single-critic)")
    args = p.parse_args()
    if args.out is None:
        args.out = ("fqe_vs_true_single.json" if args.single_critic
                    else "fqe_vs_true.json")
    torch.set_num_threads(args.threads)

    buffer = load_minari(DATASET_ID)
    dataset = minari.load_dataset(DATASET_ID)
    env = dataset.recover_environment(max_episode_steps=args.horizon)
    seeds = np.arange(5000, 5000 + args.starts)
    results = {}

    for name in args.runs:
        torch.manual_seed(0)  # per run, so a run's FQE fit ignores the order
        ck = torch.load(os.path.join(CODE_DIR, "runs", name, "checkpoint.pt"))
        actor = Actor(6, 2)
        actor.load_state_dict(ck["actor"])
        actor.eval()

        s0, g_true = true_returns(env, actor, seeds, args.horizon)
        fqe = Critic(6, 2)
        if args.single_critic:
            fqe_single_critic(actor, buffer, fqe, args.fqe_steps)
        else:
            fitted_q_evaluation(actor, buffer, fqe, steps=args.fqe_steps,
                                  gamma=GAMMA)
        with torch.no_grad():
            st = torch.as_tensor(s0)
            a0, _ = actor(st, deterministic=True)
            q1, q2 = fqe(st, a0)
            est = q1 if args.single_critic else torch.min(q1, q2)
            g_fqe = est.squeeze(-1).numpy()

        corr = (float(np.corrcoef(g_fqe, g_true)[0, 1])
                if np.std(g_true) > 0 else None)
        results[name] = {
            "checkpoint_step": int(ck["step"]),
            "critic": "single" if args.single_critic else "twin_min",
            "fqe_steps": args.fqe_steps,
            "true_mean": float(g_true.mean()), "true_std": float(g_true.std()),
            "fqe_mean": float(g_fqe.mean()), "fqe_std": float(g_fqe.std()),
            "mean_abs_error": float(np.abs(g_fqe - g_true).mean()),
            "per_state_corr": corr,
        }
        print(name, json.dumps(results[name]), flush=True)

    out = os.path.join(CODE_DIR, "runs", args.out)
    with open(out, "w") as f:
        json.dump(results, f, indent=2)
    print("saved", out)


The cell below runs the same computation as that script's `main` for both
runs, and keeps the per-state numbers so it can plot them. FQE takes a few
minutes per run at 100,000 steps. Set `SINGLE_CRITIC = True` for the
single-critic comparison.

In the chapter's run, the twin-minimum estimate for CQL was 0.32 against a
true return of 1.27, and still rising: 0.61 after 300,000 steps. A single
critic overshot instead, to 1.59 and then 1.99, and its per-state correlation
fell from 0.75 to 0.58. Ask the exercise's questions of your own numbers: does
FQE rank the two agents correctly, does it track which start states are
better, and how close are its values to the truth?


In [ ]:
# This module's main writes the same comparison to runs/fqe_vs_true.json.
fqe_vs_true = main

FQE_STEPS, STARTS, HORIZON = 100_000, 50, 800
SINGLE_CRITIC = False  # True fits and reads q1 alone (exercise 9.2)
if FAST:
    FQE_STEPS, STARTS, HORIZON = 200, 3, 50

env = minari.load_dataset(DATASET_ID).recover_environment(
    max_episode_steps=HORIZON)
seeds = np.arange(5000, 5000 + STARTS)
per_state = {}

for name in RUNS:
    torch.manual_seed(0)  # per run, as the script does
    ck = torch.load(os.path.join(CODE_DIR, "runs", name, "checkpoint.pt"))
    policy = Actor(6, 2)
    policy.load_state_dict(ck["actor"])
    policy.eval()

    s0, g_true = true_returns(env, policy, seeds, HORIZON)
    fqe = Critic(6, 2)
    if SINGLE_CRITIC:
        fqe_single_critic(policy, buffer, fqe, FQE_STEPS)
    else:
        fitted_q_evaluation(policy, buffer, fqe, steps=FQE_STEPS, gamma=GAMMA)
    with torch.no_grad():
        st = torch.as_tensor(s0)
        q1, q2 = fqe(st, policy(st, deterministic=True)[0])
        est = q1 if SINGLE_CRITIC else torch.min(q1, q2)
    g_fqe = est.squeeze(-1).numpy()
    per_state[name] = g_true, g_fqe

    corr = (np.corrcoef(g_fqe, g_true)[0, 1] if np.std(g_true) > 0
            else float("nan"))
    print(f"{name}: step {ck['step']:,}  true {g_true.mean():.3f}  "
          f"FQE {g_fqe.mean():.3f}  "
          f"mean abs error {np.abs(g_fqe - g_true).mean():.3f}  "
          f"per-state corr {corr:.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.6))
for name, (g_true, g_fqe) in per_state.items():
    label, color, _ = STYLE[name]
    ax.scatter(g_true, g_fqe, s=22, color=color, label=label, alpha=0.8)
lo = min(min(v.min() for v in pair) for pair in per_state.values())
hi = max(max(v.max() for v in pair) for pair in per_state.values())
ax.plot([lo, hi], [lo, hi], color="#adb5bd", linestyle="--", linewidth=1,
        label="FQE = true")
ax.set_xlabel("true discounted return from the start state", fontsize=9)
ax.set_ylabel("FQE estimate", fontsize=9)
ax.set_title("FQE against the truth, one point per start state",
             fontsize=10, loc="left")
ax.grid(color="#e9ecef", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()


## Where to go next

`run(name, cql_alpha, steps)` from section 7 trains any new configuration and
resumes a shorter one, so most exercises are one line here.

- **Exercise 9.3: hyperparameter sensitivity.** `run("cql_a0.1", 0.1)`,
  `run("cql_a1", 1.0)` and `run("cql_a10", 10.0)`, then add the names to `RUNS`
  (and to `STYLE`) and rerun sections 8 to 10.
- **Exercise 9.6: reading the dataset Q-value.** Train at α = 1 for 50,000
  steps, then continue the same run to 200,000 with `run("cql_a1", 1.0,
  steps=200_000)`, which resumes it. Is a falling dataset Q-value overfitting
  or a conservative estimate settling, and is 1.0 large enough by the CQL
  penalty's rule?
- **Exercise 9.5: CQL-Lagrange.** Make α a learnable parameter in log-space,
  as chapter 6 did for the temperature, and update it to maximize α times the
  gap between the soft maximum and the dataset Q-values minus a threshold τ.
  Edit a copy of listing 9.1's cell rather than the cell itself, so the
  printed listing stays intact.
- **Exercise 9.7: a behavior-cloning baseline.** Train chapter 6's actor to
  minimize the squared error between its deterministic action and the logged
  action, with no critic, and evaluate it with `evaluate(make_eval_env(),
  actor)`. The logs come from a planner that already routes around the wall,
  so ask whether you would expect it to reach the goal.

From the repository root, the same experiments run as scripts:

```bash
python -m src.part_3_applications.ch09_cql.train_cql_vs_sac --name cql --cql-alpha 5.0
python -m src.part_3_applications.ch09_cql.train_cql_vs_sac --name sac --cql-alpha 0.0
python -m src.part_3_applications.ch09_cql.eval_fqe_vs_true cql sac
```
